# KYC onboarding: EDA and findings

**Analyst story in one line:** the onboarding decisions follow a simple, fully recoverable policy (100% reconciliation), but the identity and address verification controls are *not* part of that policy, so three in four approvals carry a failed check. That is a **control gap, not a logic error**.

This notebook profiles the raw synthetic CSVs directly with DuckDB (it never loads OCR text), recovers the rule engine, quantifies the control breach and shows the flat distributions that the dashboard refuses to over-interpret. Every number below is computed here; nothing is typed in.

*Data: 14 synthetic CSVs from `archive.zip`, read from `data/raw` (falls back to the 1,000-customer `data/sample` when the raw files are absent).*

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", None)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
if not (RAW / "kyc_cases.csv").exists():
    RAW = ROOT / "data" / "sample"
AS_OF = "2026-06-30"
con = duckdb.connect()
q = lambda sql: con.execute(sql).df()  # noqa: E731
print("reading from", RAW.relative_to(ROOT))

reading from data/raw


## 1. Profile: row counts of every file

In [2]:
rows = []
for path in sorted(RAW.glob("*.csv")):
    n = con.execute(f"SELECT COUNT(*) FROM read_csv('{path.as_posix()}', header = true)").fetchone()[0]
    rows.append({"file": path.name, "rows": n, "size_mb": round(path.stat().st_size / 1e6, 1)})
profile = pd.DataFrame(rows)
print(f"{len(profile)} files, {profile['size_mb'].sum():,.1f} MB")
profile

14 files, 623.4 MB


,file,rows,size_mb
0,aml_rules.csv,3000,0.4
1,benchmark_dataset.csv,150000,63.4
2,customer_documents.csv,500000,99.0
3,customer_profiles.csv,100000,12.1
4,ground_truth_answers.csv,500000,160.9
5,hallucinated_answers.csv,100000,38.0
6,hallucination_labels.csv,100000,6.2
7,hallucination_training_dataset.csv,150000,8.4
8,hallucination_training_dataset_v2.csv,150000,61.8
9,kyc_cases.csv,100000,10.7


In [3]:
con.execute(f"CREATE TABLE p AS SELECT * FROM read_csv('{(RAW / 'customer_profiles.csv').as_posix()}', header = true)")
con.execute(f"CREATE TABLE c AS SELECT * FROM read_csv('{(RAW / 'kyc_cases.csv').as_posix()}', header = true)")
con.execute(
    "CREATE TABLE d AS SELECT DocumentID, CustomerID, DocumentType, IssueCountry, ExpiryDate, Verified, Confidence "
    f"FROM read_csv('{(RAW / 'customer_documents.csv').as_posix()}', header = true)"
)
q("SELECT ExpectedDecision AS decision, COUNT(*) AS cases, ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct "
  "FROM c GROUP BY 1 ORDER BY 2 DESC")

,decision,cases,pct
0,Approve,63001,63.0
1,Enhanced Due Diligence,23121,23.1
2,Reject,7877,7.9
3,Manual Review,4577,4.6
4,Pending Documents,1424,1.4


In [4]:
q("""
SELECT
  COUNT(*) FILTER (WHERE PEPStatus) AS pep_yes,
  COUNT(*) FILTER (WHERE SanctionStatus) AS sanction_yes,
  COUNT(*) FILTER (WHERE RiskCategory = 'High') AS risk_high,
  COUNT(*) FILTER (WHERE RiskCategory = 'Medium') AS risk_medium,
  COUNT(*) FILTER (WHERE RiskCategory = 'Low') AS risk_low
FROM c
""")

,pep_yes,sanction_yes,risk_high,risk_medium,risk_low
0,20030,7877,35575,36671,27754


## 2. Documents: VerifiedDocuments is consistent with the document table

In [5]:
q("""
WITH v AS (SELECT CustomerID, COUNT(*) AS docs, COUNT(*) FILTER (WHERE Verified) AS verified FROM d GROUP BY 1)
SELECT COUNT(*) AS customers,
       ROUND(100.0 * AVG((c.VerifiedDocuments = v.verified)::INT), 2) AS pct_consistent,
       MIN(v.docs) AS min_docs, MAX(v.docs) AS max_docs
FROM c JOIN v USING (CustomerID)
""")

,customers,pct_consistent,min_docs,max_docs
0,100000,100.0,5,5


In [6]:
q("SELECT VerifiedDocuments, COUNT(*) AS cases FROM c GROUP BY 1 ORDER BY 1")

,VerifiedDocuments,cases
0,1,5
1,2,126
2,3,2068
3,4,20414
4,5,77387


## 3. Recovering the decision policy

Cross-tabulating the labelled decision against the screening flags shows a strict precedence: sanction, then PEP, then High risk, then fewer than 4 verified documents.

In [7]:
policy = q("""
SELECT SanctionStatus AS sanctioned, PEPStatus AS pep, RiskCategory = 'High' AS high_risk,
       VerifiedDocuments < 4 AS under_4_docs, ExpectedDecision AS decision, COUNT(*) AS cases
FROM c GROUP BY ALL ORDER BY sanctioned DESC, pep DESC, high_risk DESC, under_4_docs DESC, cases DESC
""")
policy.groupby(["sanctioned", "pep", "high_risk", "under_4_docs"])["decision"].agg(lambda s: sorted(set(s)))

sanctioned  pep    high_risk  under_4_docs
False       False  False      False                                         [Approve]
                              True                                [Pending Documents]
                   True       False           [Enhanced Due Diligence, Manual Review]
                              True            [Enhanced Due Diligence, Manual Review]
            True   True       False                          [Enhanced Due Diligence]
                              True                           [Enhanced Due Diligence]
True        False  True       False                                          [Reject]
                              True                                           [Reject]
            True   True       False                                          [Reject]
                              True                                           [Reject]
Name: decision, dtype: object

In [8]:
check = q("""
SELECT CASE WHEN SanctionStatus THEN 'Reject'
            WHEN PEPStatus THEN 'Enhanced Due Diligence'
            WHEN RiskCategory = 'High' THEN 'Escalate (EDD/Manual)'
            WHEN VerifiedDocuments < 4 THEN 'Pending Documents'
            ELSE 'Approve' END AS engine,
       ExpectedDecision AS decision, COUNT(*) AS cases
FROM c GROUP BY ALL
""")
check["agrees"] = [
    d in ("Enhanced Due Diligence", "Manual Review") if e == "Escalate (EDD/Manual)" else d == e
    for e, d in zip(check["engine"], check["decision"], strict=True)
]
agreement = check.loc[check["agrees"], "cases"].sum() / check["cases"].sum()
print(f"Rule engine reconciles {agreement:.1%} of {check['cases'].sum():,} labelled decisions")
check.sort_values("cases", ascending=False)

Rule engine reconciles 100.0% of 100,000 labelled decisions


,engine,decision,cases,agrees
1,Approve,Approve,63001,True
3,Enhanced Due Diligence,Enhanced Due Diligence,18475,True
0,Reject,Reject,7877,True
4,Escalate (EDD/Manual),Enhanced Due Diligence,4646,True
5,Escalate (EDD/Manual),Manual Review,4577,True
2,Pending Documents,Pending Documents,1424,True


In [9]:
esc = q("SELECT ExpectedDecision, COUNT(*) AS cases FROM c WHERE NOT SanctionStatus AND NOT PEPStatus "
        "AND RiskCategory = 'High' GROUP BY 1")
esc["share"] = (esc["cases"] / esc["cases"].sum()).round(4)
esc  # the EDD vs Manual Review split for High-risk escalations is ~50/50 and unexplained

,ExpectedDecision,cases,share
0,Enhanced Due Diligence,4646,0.5037
1,Manual Review,4577,0.4963


## 4. The headline: approvals despite failed verification

In [10]:
breach = q("""
SELECT COUNT(*) AS approvals,
       COUNT(*) FILTER (WHERE NOT p.IdentityVerified) AS identity_unverified,
       COUNT(*) FILTER (WHERE NOT p.AddressVerified) AS address_unverified,
       COUNT(*) FILTER (WHERE NOT p.IdentityVerified OR NOT p.AddressVerified) AS id_or_address_unverified
FROM c JOIN p USING (CustomerID) WHERE c.ExpectedDecision = 'Approve'
""")
b = breach.iloc[0]
print(f"{b.id_or_address_unverified:,} of {b.approvals:,} approvals ({b.id_or_address_unverified / b.approvals:.1%}) "
      "have identity or address unverified")
breach

47,157 of 63,001 approvals (74.9%) have identity or address unverified


,approvals,identity_unverified,address_unverified,id_or_address_unverified
0,63001,31377,31461,47157


In [11]:
q("""
SELECT c.ExpectedDecision AS decision,
       ROUND(AVG(p.IdentityVerified::INT), 3) AS identity_verified_rate,
       ROUND(AVG(p.AddressVerified::INT), 3) AS address_verified_rate
FROM c JOIN p USING (CustomerID) GROUP BY 1 ORDER BY 1
""")  # ~50% in every decision: the verification flags play no role in the decision

,decision,identity_verified_rate,address_verified_rate
0,Approve,0.502,0.501
1,Enhanced Due Diligence,0.502,0.500
2,Manual Review,0.504,0.485
3,Pending Documents,0.529,0.510
4,Reject,0.503,0.502


## 5. Periodic-review backlog

In [12]:
q(f"""
SELECT COUNT(*) AS customers,
       COUNT(*) FILTER (WHERE date_diff('day', LastUpdated, DATE '{AS_OF}') > 365) AS overdue_at_as_of,
       COUNT(*) FILTER (WHERE date_diff('day', LastUpdated, (SELECT MAX(LastUpdated) FROM p)) > 365)
           AS overdue_at_extract_date,
       MAX(LastUpdated) AS extract_date
FROM p
""")

,customers,overdue_at_as_of,overdue_at_extract_date,extract_date
0,100000,50620,49828,2026-06-24


## 6. Flat distributions (the dashboard says so instead of inventing a story)

In [13]:
q("""
SELECT Country, COUNT(*) AS cases, ROUND(100 * AVG(SanctionStatus::INT), 2) AS sanction_pct,
       ROUND(100 * AVG(PEPStatus::INT), 2) AS pep_pct, ROUND(100 * AVG((RiskCategory = 'High')::INT), 2) AS high_pct
FROM c GROUP BY 1 ORDER BY 1
""")

,Country,cases,sanction_pct,pep_pct,high_pct
0,India,20017,8.16,20.34,36.30
1,Iran,19952,7.63,19.62,35.39
2,Russia,19850,7.66,20.38,35.43
3,UK,20060,7.87,19.98,35.32
4,USA,20121,8.06,19.83,35.43


In [14]:
q("SELECT Occupation, MIN(Income) AS min_income, ROUND(AVG(Income)) AS mean_income, MAX(Income) AS max_income "
  "FROM p GROUP BY 1 ORDER BY 1")

,Occupation,min_income,mean_income,max_income
0,Accountant,200817,5095860.0,9999042
1,Banker,200578,5145410.0,9999893
2,Business Owner,200788,5125949.0,9998918
3,Consultant,200214,5063831.0,9999726
4,Doctor,200703,5060580.0,9997666
5,Government Employee,200012,5076413.0,9999387
6,Lawyer,201318,5094706.0,9999882
7,Software Engineer,200597,5110109.0,9998447
8,Student,200723,5082543.0,9999911
9,Teacher,201281,5070689.0,9998331


In [15]:
q("""
SELECT DocumentType, ROUND(100 * AVG((NOT Verified)::INT), 2) AS fail_pct,
       ROUND(AVG(Confidence), 4) AS mean_conf, MIN(ExpiryDate) AS first_expiry, MAX(ExpiryDate) AS last_expiry
FROM d GROUP BY 1 ORDER BY 1
""")

,DocumentType,fail_pct,mean_conf,first_expiry,last_expiry
0,Aadhaar,4.90,0.8952,2027-06-25,2036-06-23
1,Driving License,4.92,0.8952,2027-06-25,2036-06-23
2,PAN,5.02,0.8952,2027-06-25,2036-06-23
3,Passport,5.03,0.8952,2027-06-25,2036-06-23
4,Utility Bill,5.08,0.8948,2027-06-25,2036-06-23


## 7. Data traps found while profiling

In [16]:
q("""
SELECT (SELECT COUNT(*) - COUNT(DISTINCT DocumentID) FROM d) AS surplus_document_id_rows,
       (SELECT COUNT(*) - COUNT(DISTINCT (CustomerID, DocumentType)) FROM d) AS duplicate_customer_doc_type,
       (SELECT COUNT(*) FROM p WHERE Age <> date_sub('year', DOB, (SELECT MAX(LastUpdated) FROM p))) AS age_mismatch_at_extract,
       (SELECT COUNT(*) FROM p WHERE Age <> date_sub('year', DOB, DATE '2026-06-30')) AS age_mismatch_at_as_of,
       (SELECT ROUND(100 * AVG((RiskCategory = 'High')::INT), 1) FROM c WHERE PEPStatus OR SanctionStatus)
           AS pct_pep_or_sanctioned_that_are_high_risk
""")

,surplus_document_id_rows,duplicate_customer_doc_type,age_mismatch_at_extract,age_mismatch_at_as_of,pct_pep_or_sanctioned_that_are_high_risk
0,13533,0,0,1693,100.0


## 8. AI copilot benchmark: templated answers

In [17]:
bench = RAW / "benchmark_dataset.csv"
q(f"""
SELECT COUNT(*) AS answers,
       COUNT(DISTINCT regexp_replace(GeneratedAnswer, 'C[0-9]+', 'C#', 'g')) AS generated_templates,
       ROUND(100 * AVG(Hallucinated), 2) AS hallucination_pct
FROM read_csv('{bench.as_posix()}', header = true)
""")

,answers,generated_templates,hallucination_pct
0,150000,25,66.67


In [18]:
q(f"""
SELECT regexp_replace(Question, 'C[0-9]+', 'C#') AS question_template, COUNT(*) AS answers,
       ROUND(100 * AVG(Hallucinated), 2) AS hallucination_pct
FROM read_csv('{bench.as_posix()}', header = true) GROUP BY 1 ORDER BY 1
""")

,question_template,answers,hallucination_pct
0,Can customer C# be onboarded?,29855,66.69
1,Can the account for customer C# be approved?,29978,66.80
2,Should Enhanced Due Diligence be applied to customer C#?,30327,66.50
3,What is the AML risk associated with customer C#?,30114,66.50
4,What verification steps are required for customer C#?,29726,66.86


## 9. Conclusions

1. **Policy is sound and provable:** a five-rule precedence engine reproduces every labelled decision (EDD and Manual Review both count for High-risk escalations, which the data splits about 50/50 with no explanatory field).
2. **Controls are not enforced:** identity and address verification are ~50/50 in every decision class, so roughly three in four approvals carry at least one failed verification flag. Recommendation: make verification a hard gate before approval.
3. **Periodic review is behind** for about half of customers, at the same rate for every risk tier.
4. **Nothing else carries signal:** country, occupation, income and age are flat; the dashboard labels these charts as flat.
5. **Operational timing does not exist in the source** and is simulated (seed 42) in the warehouse; it is disclosed wherever it is shown.
6. **The AI benchmark is templated**, so detector scores near 100% are an artefact of the data, not evidence of trustworthiness.